In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install sentence-transformers bert-score --quiet

import pandas as pd, numpy as np, torch, ast, os, json
from sentence_transformers import SentenceTransformer, InputExample, CrossEncoder
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from bert_score import score as bert_score_compute
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f'GPU disponibil: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memorie GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

GPU disponibil: True
Device: Tesla T4
Memorie GPU: 14.6 GB


/tmp/ipykernel_175/3081942614.py:5: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss


In [2]:
INPUT_PATH  = '/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv'
OUTPUT_PATH = '/kaggle/working/'

df = pd.read_csv(INPUT_PATH)
print(f'Filme incarcate: {len(df):,}')
print(f'Coloane: {df.columns.tolist()}')

df['overview']         = df['overview'].fillna('').astype(str).str.strip()
df['tagline']          = df['tagline'].fillna('').astype(str).str.strip()
df['review_summary']   = df['review_summary'].fillna('').astype(str).str.strip().replace('nan', '')
df['overview_summary'] = df['overview_summary'].fillna('').astype(str).str.strip().replace('nan', '')
df['genres']           = df['genres'].fillna('').astype(str).str.strip()
df['keywords']         = df['keywords'].fillna('').astype(str).str.strip()
df['cast']             = df['cast'].fillna('').astype(str).str.strip()
df['director']         = df['director'].fillna('').astype(str).str.strip()

print()
for col in ['overview', 'overview_summary', 'review_summary', 'tagline', 'keywords']:
    n   = (df[col] != '').sum()
    avg = df[col].str.split().str.len().mean()
    print(f'  {col:<22} acoperire: {n:>6,} ({n/len(df)*100:.1f}%)   medie: ~{avg:.0f} cuvinte')

Filme incarcate: 40,197
Coloane: ['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review', 'overview_summary', 'input_review', 'review_summary']

  overview               acoperire: 40,197 (100.0%)   medie: ~47 cuvinte
  overview_summary       acoperire: 40,197 (100.0%)   medie: ~26 cuvinte
  review_summary         acoperire: 11,644 (29.0%)   medie: ~11 cuvinte
  tagline                acoperire: 40,197 (100.0%)   medie: ~9 cuvinte
  keywords               acoperire: 40,197 (100.0%)   medie: ~10 cuvinte


In [3]:
def extract_cast_names(cast_raw, max_actors=4):
    if not isinstance(cast_raw, str) or not cast_raw.strip():
        return ''
    try:
        actors = ast.literal_eval(cast_raw)
        if isinstance(actors, list):
            names = [a['name'] for a in actors[:max_actors]
                     if isinstance(a, dict) and a.get('name')]
            return ', '.join(names)
    except Exception:
        pass
    return ''


def extract_keywords(kw_raw, max_kw=10):
    if not isinstance(kw_raw, str) or not kw_raw.strip():
        return ''
    try:
        parsed = ast.literal_eval(kw_raw)
        if isinstance(parsed, list):
            return ', '.join(str(k) for k in parsed[:max_kw])
    except Exception:
        pass
    return ', '.join(w.strip() for w in kw_raw.split(',')[:max_kw])


def parse_genres(genres_raw):
    if not isinstance(genres_raw, str) or not genres_raw.strip():
        return []
    try:
        parsed = ast.literal_eval(genres_raw)
        if isinstance(parsed, list):
            return [str(g).lower().strip() for g in parsed if g]
    except Exception:
        pass
    return [g.strip().lower() for g in genres_raw.split(',') if g.strip()]


print('Functii ajutatoare definite.')

Functii ajutatoare definite.


In [4]:
def clean_overview_summary(title, ovs):
    t = str(title).strip()
    s = str(ovs).strip()
    if len(t) > 3 and s.lower().startswith(t.lower()):
        s = s[len(t):].lstrip('. \n').strip()
    return s if s else str(ovs).strip()


def build_doc_text_v5(row):
    parts = [str(row.get('title', '')).strip() + '.']

    ov = str(row.get('overview', '')).replace('nan', '').strip()
    if ov:
        parts.append('Plot: ' + ov)

    rev = str(row.get('review_summary', '')).replace('nan', '').strip()
    if rev:
        parts.append('Critics: ' + rev)

    genres = str(row.get('genres', '')).replace('nan', '').strip()
    if genres:
        parts.append('Genres: ' + genres)

    kw = extract_keywords(row.get('keywords', ''), 10)
    if kw:
        parts.append('Keywords: ' + kw)

    cast = extract_cast_names(row.get('cast', ''), 4)
    if cast:
        parts.append('Cast: ' + cast)

    return ' '.join(parts)


def build_doc_text_no_review(row):
    parts = [str(row.get('title', '')).strip() + '.']

    ov = str(row.get('overview', '')).replace('nan', '').strip()
    if ov:
        parts.append('Plot: ' + ov)

    genres = str(row.get('genres', '')).replace('nan', '').strip()
    if genres:
        parts.append('Genres: ' + genres)

    kw = extract_keywords(row.get('keywords', ''), 10)
    if kw:
        parts.append('Keywords: ' + kw)

    cast = extract_cast_names(row.get('cast', ''), 4)
    if cast:
        parts.append('Cast: ' + cast)

    return ' '.join(parts)


df['doc_text']           = df.apply(build_doc_text_v5, axis=1)
df['doc_text_no_review'] = df.apply(build_doc_text_no_review, axis=1)

mask     = (df['tagline'] != '') & (df['doc_text'] != '')
df_valid = df[mask].copy().reset_index(drop=True)

print(f'Filme valide: {len(df_valid):,}')

real_leak = df_valid.apply(
    lambda r: len(str(r['tagline'])) > 15
              and str(r['tagline']).lower() in str(r['doc_text']).lower(),
    axis=1
).sum()
print(f'Leakage (tagline in doc_text): {real_leak} cazuri (~0.5% acceptabil)')

kw_cov = (df_valid['keywords'] != '').sum()
ov_cov = (df_valid['overview'] != '').sum()
print(f'Filme cu overview:  {ov_cov:,} ({ov_cov/len(df_valid)*100:.1f}%)')
print(f'Filme cu keywords:  {kw_cov:,} ({kw_cov/len(df_valid)*100:.1f}%)')

ex = df_valid[df_valid['keywords'] != ''].iloc[0]
print(f'\n--- Exemplu doc_text V5: {ex["title"]} ---')
print(ex['doc_text'])

Filme valide: 40,197
Leakage (tagline in doc_text): 248 cazuri (~0.5% acceptabil)
Filme cu overview:  40,197 (100.0%)
Filme cu keywords:  40,197 (100.0%)

--- Exemplu doc_text V5: Toy Story ---
Toy Story. Plot: Led by Woody, Andys toys live happily in his room until Andys birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andys heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Critics: Tom Hanks leads a strong cast. Tim Allen is great, too, as Buzz Lightyear. Don Rickles (Mr. Potato Head), Wallace Shawn (Rex) and John Ratzenberger (Hamm) also bring fun. Genres: ['Family', 'Comedy', 'Animation', 'Adventure'] Keywords: rescue, friendship, mission, jealousy, villain, bullying, elementary school, rivalry, anthropomorphism, friends Cast: Tom Hanks, Tim Allen, Don Rickles, Jim Varney


In [5]:
from transformers import AutoTokenizer

tok    = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5')
sample = df_valid['doc_text'].sample(1000, random_state=42)
lengths = [len(tok(t, truncation=False)['input_ids']) for t in sample]

with_rev = df_valid[df_valid['review_summary'] != '']['doc_text'].sample(
    min(500, (df_valid['review_summary'] != '').sum()), random_state=42)
no_rev   = df_valid[df_valid['review_summary'] == '']['doc_text'].sample(
    min(500, (df_valid['review_summary'] == '').sum()), random_state=42)

len_with = [len(tok(t, truncation=False)['input_ids']) for t in with_rev]
len_no   = [len(tok(t, truncation=False)['input_ids']) for t in no_rev]

print(f'doc_text V5 CU review   — Medie: {np.mean(len_with):.0f} | Max: {max(len_with)} | >512: {sum(l>512 for l in len_with)}/{len(len_with)}')
print(f'doc_text V5 FARA review — Medie: {np.mean(len_no):.0f}   | Max: {max(len_no)}   | >512: {sum(l>512 for l in len_no)}/{len(len_no)}')
print(f'Global                  — Medie: {np.mean(lengths):.0f}  | 99p: {np.percentile(lengths,99):.0f}   | >512: {sum(l>512 for l in lengths)}/{len(lengths)}')

doc_text V5 CU review   — Medie: 166 | Max: 315 | >512: 0/500
doc_text V5 FARA review — Medie: 109   | Max: 272   | >512: 0/500
Global                  — Medie: 126  | 99p: 239   | >512: 0/1000


In [6]:
BGE_QUERY_PREFIX = 'Represent this sentence: '

train_df, eval_df = train_test_split(df_valid, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
eval_df  = eval_df.reset_index(drop=True)
print(f'Training: {len(train_df):,} filme')
print(f'Eval:     {len(eval_df):,} filme')

train_examples = [
    InputExample(texts=[BGE_QUERY_PREFIX + row['tagline'], row['doc_text']])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

train_examples_no_review = [
    InputExample(texts=[BGE_QUERY_PREFIX + row['tagline'], row['doc_text_no_review']])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

n_with = sum(1 for _, r in train_df.iterrows() if r['review_summary'] not in ('', 'nan'))
print(f'\nExemple principale:  {len(train_examples):,}')
print(f'  - cu review:       {n_with:,} ({n_with/len(train_df)*100:.1f}%)')
print(f'  - fara review:     {len(train_df)-n_with:,}')
print(f'\nExemplu pereche:')
ex = train_examples[0]
print(f'  Query: {ex.texts[0]}')
print(f'  Doc:   {ex.texts[1][:250]}...')

Training: 36,177 filme
Eval:     4,020 filme

Exemple principale:  35,788
  - cu review:       10,438 (28.9%)
  - fara review:     25,739

Exemplu pereche:
  Query: Represent this sentence: She loved and trusted him until he cut off her head.
  Doc:   Savage Intruder. Plot: An enigmatic young man manipulates his way into working at the decaying mansion of a once prolific, but now reclusive and alcoholic, movie star named Katharine Packard. While the rest of the house staff become suspicious of Vic...


In [7]:
MODEL_NAME = 'BAAI/bge-base-en-v1.5'
OUTPUT_V5A = OUTPUT_PATH + 'sbert_v5a'
OUTPUT_V5B = OUTPUT_PATH + 'sbert_v5b'
device     = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

import gc
torch.cuda.empty_cache()
gc.collect()

sbert = SentenceTransformer(MODEL_NAME, device=str(device))

print(f'Model: {MODEL_NAME}')
print(f'Device: {device}')
print(f'Embedding dim: {sbert.encode(["test"]).shape[1]}')

total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
free_mem  = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3
print(f'GPU: {total_mem:.1f} GB total | {free_mem:.1f} GB libera')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: BAAI/bge-base-en-v1.5
Device: cuda:0
Embedding dim: 768
GPU: 14.6 GB total | 14.1 GB libera


In [8]:
import gc
torch.cuda.empty_cache()
gc.collect()

sbert   = sbert.to(device)
_tok    = sbert.tokenizer

def tokenize_batch(texts, max_length=256):
    return _tok(texts, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt')

def get_embeddings(encoded):
    encoded = {k: v.to(device) for k, v in encoded.items()}
    out = sbert(encoded)
    return F.normalize(out['sentence_embedding'], p=2, dim=1)

def mnr_loss_fn(emb_a, emb_p, scale=50.0):
    scores = torch.mm(emb_a, emb_p.T) * scale
    labels = torch.arange(len(emb_a), device=device)
    return F.cross_entropy(scores, labels)

def simple_collate(examples):
    return [e.texts[0] for e in examples], [e.texts[1] for e in examples]

BATCH_SIZE  = 64
EPOCHS      = 3

train_dl     = DataLoader(train_examples, batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=simple_collate)
total_steps  = len(train_dl) * EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(sbert.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler    = GradScaler('cuda')

print(f'Antrenare V5a: model={MODEL_NAME} | batch={BATCH_SIZE} | epoci={EPOCHS}')
print(f'  total steps={total_steps:,} | warmup={warmup_steps:,}')

for epoch in range(EPOCHS):
    sbert.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_dl), total=len(train_dl),
                desc=f'Epoch {epoch+1}/{EPOCHS}')
    for step, (anchors, positives) in pbar:
        enc_a = tokenize_batch(anchors)
        enc_p = tokenize_batch(positives)

        with autocast('cuda'):
            emb_a = get_embeddings(enc_a)
            emb_p = get_embeddings(enc_p)
            loss  = mnr_loss_fn(emb_a, emb_p)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(sbert.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    print(f'Epoch {epoch+1}/{EPOCHS} — Loss mediu: {total_loss/len(train_dl):.4f}')
    sbert.save(OUTPUT_PATH + f'sbert_v5a_epoch{epoch+1}')

sbert.save(OUTPUT_V5A)
print(f'\nV5a salvat: {OUTPUT_V5A}')

Antrenare V5a: model=BAAI/bge-base-en-v1.5 | batch=64 | epoci=3
  total steps=1,680 | warmup=168


Epoch 1/3:   0%|          | 0/560 [00:00<?, ?it/s]

Epoch 1/3 — Loss mediu: 1.9522


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2/3:   0%|          | 0/560 [00:00<?, ?it/s]

...

Epoch 3/3:   0%|          | 0/560 [00:00<?, ?it/s]

Epoch 3/3 — Loss mediu: 1.2849


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


V5a salvat: /kaggle/working/sbert_v5a


In [9]:
BATCH_EMB = 256

queries_prefixed = [BGE_QUERY_PREFIX + t for t in df_valid['tagline'].tolist()]
docs             = df_valid['doc_text'].tolist()
docs_no_review   = df_valid['doc_text_no_review'].tolist()

sbert_base = SentenceTransformer(MODEL_NAME, device=str(device))
sbert_v5a  = SentenceTransformer(OUTPUT_V5A,  device=str(device))

print('Generare embeddings Baseline BGE...')
q_emb_base = sbert_base.encode(queries_prefixed, batch_size=BATCH_EMB,
                                show_progress_bar=True, convert_to_numpy=True,
                                normalize_embeddings=True)
d_emb_base = sbert_base.encode(docs, batch_size=BATCH_EMB,
                                show_progress_bar=True, convert_to_numpy=True,
                                normalize_embeddings=True)

print('\nGenerare embeddings V5a (fine-tuned, cu keywords)...')
q_emb_v5a = sbert_v5a.encode(queries_prefixed, batch_size=BATCH_EMB,
                               show_progress_bar=True, convert_to_numpy=True,
                               normalize_embeddings=True)
d_emb_v5a = sbert_v5a.encode(docs, batch_size=BATCH_EMB,
                               show_progress_bar=True, convert_to_numpy=True,
                               normalize_embeddings=True)

print('\nGenerare embeddings V5a ablatie (fara review)...')
d_emb_v5a_norev = sbert_v5a.encode(docs_no_review, batch_size=BATCH_EMB,
                                    show_progress_bar=True, convert_to_numpy=True,
                                    normalize_embeddings=True)

np.save(OUTPUT_PATH + 'q_emb_base.npy',      q_emb_base)
np.save(OUTPUT_PATH + 'd_emb_base.npy',      d_emb_base)
np.save(OUTPUT_PATH + 'q_emb_v5a.npy',       q_emb_v5a)
np.save(OUTPUT_PATH + 'd_emb_v5a.npy',       d_emb_v5a)
np.save(OUTPUT_PATH + 'd_emb_v5a_norev.npy', d_emb_v5a_norev)
print('\nEmbeddings salvate!')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generare embeddings Baseline BGE...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Generare embeddings V5a ablatie (fara review)...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Embeddings salvate!


In [10]:
df_valid['genres_list'] = df_valid['genres'].apply(parse_genres)

genre_to_indices = {}
for i, genres in enumerate(df_valid['genres_list']):
    for g in genres:
        genre_to_indices.setdefault(g, []).append(i)

print(f'Genuri unice: {len(genre_to_indices)}')


def evaluate_full_pool(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20],
                       batch_size=512, label=''):
    N    = len(query_emb)
    hits = {k: 0 for k in k_values}
    max_k = max(k_values)

    for start in range(0, N, batch_size):
        end    = min(start + batch_size, N)
        scores = cosine_similarity(query_emb[start:end], doc_emb)
        top_k  = np.argpartition(scores, -max_k, axis=1)[:, -max_k:]

        for i, global_idx in enumerate(range(start, end)):
            top_sorted = top_k[i][np.argsort(scores[i][top_k[i]])[::-1]]
            for k in k_values:
                if global_idx in top_sorted[:k]:
                    hits[k] += 1

        if start % 10000 == 0 and start > 0:
            print(f'  {label} {start}/{N}...')

    return {k: hits[k] / N for k in k_values}


def evaluate_genre_filtered(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20],
                             sample_n=3000, min_pool=30, label=''):
    np.random.seed(42)
    indices    = np.random.choice(len(query_emb), min(sample_n, len(query_emb)), replace=False)
    hits       = {k: 0 for k in k_values}
    pool_sizes = []

    for idx in indices:
        pool = set()
        for g in df_valid['genres_list'].iloc[idx]:
            pool.update(genre_to_indices.get(g, []))
        pool.add(idx)
        if len(pool) < min_pool:
            pool = set(range(len(df_valid)))

        pool_arr  = np.array(sorted(pool))
        pool_sizes.append(len(pool_arr))
        scores    = cosine_similarity(query_emb[idx:idx+1], doc_emb[pool_arr])[0]
        local_idx = int(np.where(pool_arr == idx)[0][0])
        ranked    = np.argsort(scores)[::-1]
        rank      = int(np.where(ranked == local_idx)[0][0]) + 1

        for k in k_values:
            if rank <= k:
                hits[k] += 1

    n = len(indices)
    print(f'  Pool mediu per query: {np.mean(pool_sizes):.0f} filme')
    return {k: hits[k] / n for k in k_values}


print('Functii de evaluare definite.')

Genuri unice: 19
Functii de evaluare definite.


In [11]:
results = {}

print('FULL POOL')

print('\nBaseline BGE (fara fine-tuning):')
results['base_full'] = evaluate_full_pool(q_emb_base, d_emb_base, label='base')
print(f'  {results["base_full"]}')

print('\nV5a (BGE fine-tuned, cu keywords):')
results['v5a_full'] = evaluate_full_pool(q_emb_v5a, d_emb_v5a, label='v5a')
print(f'  {results["v5a_full"]}')

print('\nV5a Ablatie (fara review):')
results['v5a_norev_full'] = evaluate_full_pool(q_emb_v5a, d_emb_v5a_norev, label='v5a-norev')
print(f'  {results["v5a_norev_full"]}')

print('\nGENRE-FILTERED POOL')

print('\nBaseline BGE:')
results['base_genre'] = evaluate_genre_filtered(q_emb_base, d_emb_base)
print(f'  {results["base_genre"]}')

print('\nV5a fine-tuned:')
results['v5a_genre'] = evaluate_genre_filtered(q_emb_v5a, d_emb_v5a)
print(f'  {results["v5a_genre"]}')

with open(OUTPUT_PATH + 'results_v5a.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nRezultate V5a salvate!')

FULL POOL

Baseline BGE (fara fine-tuning):
  {1: 0.06988083687837401, 3: 0.1044605318804886, 5: 0.12182501181680225, 10: 0.14856830111699879, 20: 0.18098365549667886}

V5a (BGE fine-tuned, cu keywords):
  {1: 0.11898897927706048, 3: 0.18108316541035402, 5: 0.21538920814986193, 10: 0.2711893922432022, 20: 0.3363435082220066}

V5a Ablatie (fara review):
  {1: 0.118342164838172, 3: 0.1806851257556534, 5: 0.21322486752742742, 10: 0.26802995248401623, 20: 0.3318655621066249}

GENRE-FILTERED POOL

Baseline BGE:
  Pool mediu per query: 15835 filme
  {1: 0.09533333333333334, 3: 0.13133333333333333, 5: 0.152, 10: 0.18666666666666668, 20: 0.22266666666666668}

V5a fine-tuned:
  Pool mediu per query: 15835 filme
  {1: 0.13466666666666666, 3: 0.19233333333333333, 5: 0.217, 10: 0.259, 20: 0.31366666666666665}

Rezultate V5a salvate!


In [12]:
CE_MODEL = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
print(f'Incarcare cross-encoder: {CE_MODEL}')
cross_encoder = CrossEncoder(CE_MODEL, device=str(device))
print('Cross-encoder gata.\n')

HN_STRATEGY  = 'same_genre'

HN_SAMPLE    = 8000
HN_PER_QUERY = 5
MINE_TOPK    = 100

train_taglines_prefixed = [
    BGE_QUERY_PREFIX + str(row['tagline'])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]
train_docs = [
    str(row['doc_text'])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]
train_genres_list = [
    parse_genres(str(row['genres']))
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

print('Encoding query embeddings pentru training set...')
q_emb_train = sbert_v5a.encode(
    train_taglines_prefixed, batch_size=256,
    show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True
)
print('Encoding doc embeddings pentru training set...')
d_emb_train = sbert_v5a.encode(
    train_docs, batch_size=256,
    show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True
)

np.random.seed(42)
sample_indices = np.random.choice(
    len(train_taglines_prefixed),
    min(HN_SAMPLE, len(train_taglines_prefixed)),
    replace=False
)

print(f'\nMining CE hard negatives (strategie: {HN_STRATEGY}):')
print(f'  Sample: {len(sample_indices):,} query-uri | top-{MINE_TOPK} bi-enc | {HN_PER_QUERY} HN/query')
print(f'  Predictii CE totale: ~{len(sample_indices)*MINE_TOPK:,}\n')

hard_neg_examples = []
skipped = 0

def hn_filter(genres_q, genres_c, strategy):
    common = len(set(genres_q) & set(genres_c))
    if strategy == 'cross_genre':
        return common < 2
    elif strategy == 'same_genre':
        return common >= 1
    elif strategy == 'mixed':
        return True
    return True

for i in tqdm(sample_indices, desc='CE Hard Negative Mining'):
    scores = cosine_similarity(q_emb_train[i:i+1], d_emb_train)[0]
    scores[i] = -1.0
    top_idx = np.argsort(scores)[::-1][:MINE_TOPK]

    query_raw = train_taglines_prefixed[i].replace(BGE_QUERY_PREFIX, '')
    pairs     = [(query_raw, train_docs[j]) for j in top_idx]
    ce_scores = cross_encoder.predict(pairs, batch_size=64)

    ce_ranked = np.argsort(ce_scores)[::-1]
    genres_q  = train_genres_list[i]

    hard_negs      = []
    same_hn_count  = 0
    cross_hn_count = 0

    for rank in ce_ranked:
        idx      = top_idx[rank]
        genres_c = train_genres_list[idx]
        common   = len(set(genres_q) & set(genres_c))

        if HN_STRATEGY == 'mixed':
            target_same  = HN_PER_QUERY // 2
            target_cross = HN_PER_QUERY - target_same
            if common >= 1 and same_hn_count < target_same:
                hard_negs.append(train_docs[idx])
                same_hn_count += 1
            elif common == 0 and cross_hn_count < target_cross:
                hard_negs.append(train_docs[idx])
                cross_hn_count += 1
        else:
            if not hn_filter(genres_q, genres_c, HN_STRATEGY):
                skipped += 1
                continue
            hard_negs.append(train_docs[idx])

        if len(hard_negs) >= HN_PER_QUERY:
            break

    for hn in hard_negs:
        hard_neg_examples.append(
            InputExample(texts=[
                train_taglines_prefixed[i],
                train_docs[i],
                hn
            ])
        )

print(f'\nHard negative triplete generate: {len(hard_neg_examples):,}')
print(f'Candidati sarati (filtru strategie): {skipped:,}')
if hard_neg_examples:
    ex = hard_neg_examples[0]
    print(f'\nExemplu triplet ({HN_STRATEGY}):')
    print(f'  Anchor:  {ex.texts[0]}')
    print(f'  Pozitiv: {ex.texts[1][:120]}...')
    print(f'  HardNeg: {ex.texts[2][:120]}...')

Incarcare cross-encoder: cross-encoder/ms-marco-MiniLM-L-12-v2


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Encoding doc embeddings pentru training set...


Batches:   0%|          | 0/140 [00:00<?, ?it/s]


Mining CE hard negatives (strategie: same_genre):
  Sample: 8,000 query-uri | top-100 bi-enc | 5 HN/query
  Predictii CE totale: ~800,000



CE Hard Negative Mining:   0%|          | 0/8000 [00:00<?, ?it/s]


Hard negative triplete generate: 39,606
Candidati sarati (filtru strategie): 50,914

Exemplu triplet (same_genre):
  Anchor:  Represent this sentence: A good football coach, can get away with murder
  Pozitiv: Pretty Maids All in a Row. Plot: At Oceanfront High School, female students are being targeted by an unknown serial kill...
  HardNeg: Getting Away with Murder. Plot: When the very moralistic college ethics instructor Jack Lambert finds himself living nex...


In [15]:
for var in ['cross_encoder', 'sbert', 'sbert_base', 'sbert_v5a']:
    if var in dir():
        del globals()[var]
torch.cuda.empty_cache()
gc.collect()

free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3
print(f'GPU libera dupa curatare: {free:.2f} GB')

sbert_v5b = SentenceTransformer(OUTPUT_V5A, device=str(device))
sbert_v5b = sbert_v5b.to(device)
_tok_v5b  = sbert_v5b.tokenizer

def tokenize_v5b(texts, max_length=256):
    return _tok_v5b(texts, padding=True, truncation=True,
                    max_length=max_length, return_tensors='pt')

def get_emb_v5b(encoded):
    encoded = {k: v.to(device) for k, v in encoded.items()}
    out = sbert_v5b(encoded)
    return F.normalize(out['sentence_embedding'], p=2, dim=1)

def mnr_triplet_loss(emb_a, emb_p, emb_n, scale=50.0):
    emb_docs = torch.cat([emb_p, emb_n], dim=0)
    scores   = torch.mm(emb_a, emb_docs.T) * scale
    labels   = torch.arange(len(emb_a), device=device)
    return F.cross_entropy(scores, labels)

def hn_collate(examples):
    return ([e.texts[0] for e in examples],
            [e.texts[1] for e in examples],
            [e.texts[2] for e in examples])

BATCH_HN  = 32
EPOCHS_HN = 2

hn_dl           = DataLoader(hard_neg_examples, batch_size=BATCH_HN,
                              shuffle=True, collate_fn=hn_collate)
total_steps_hn  = len(hn_dl) * EPOCHS_HN
warmup_steps_hn = int(0.05 * total_steps_hn)

optimizer_hn = AdamW(sbert_v5b.parameters(), lr=1e-5, weight_decay=0.01)
scheduler_hn = get_linear_schedule_with_warmup(optimizer_hn, warmup_steps_hn, total_steps_hn)
scaler_hn    = GradScaler('cuda')

print(f'Fine-tuning V5b cu CE hard negatives:')
print(f'  Start din: {OUTPUT_V5A}')
print(f'  batch={BATCH_HN} | triplete={len(hard_neg_examples):,} | epoci={EPOCHS_HN}')

for epoch in range(EPOCHS_HN):
    sbert_v5b.train()
    total_loss = 0.0
    optimizer_hn.zero_grad()

    pbar = tqdm(enumerate(hn_dl), total=len(hn_dl),
                desc=f'V5b Epoch {epoch+1}/{EPOCHS_HN}')
    for step, (anchors, positives, negatives) in pbar:
        enc_a = tokenize_v5b(anchors)
        enc_p = tokenize_v5b(positives)
        enc_n = tokenize_v5b(negatives)

        with autocast('cuda'):
            emb_a = get_emb_v5b(enc_a)
            emb_p = get_emb_v5b(enc_p)
            emb_n = get_emb_v5b(enc_n)
            loss  = mnr_triplet_loss(emb_a, emb_p, emb_n)

        scaler_hn.scale(loss).backward()
        scaler_hn.unscale_(optimizer_hn)
        torch.nn.utils.clip_grad_norm_(sbert_v5b.parameters(), max_norm=1.0)
        scaler_hn.step(optimizer_hn)
        scaler_hn.update()
        scheduler_hn.step()
        optimizer_hn.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    print(f'V5b Epoch {epoch+1} — Loss mediu: {total_loss/len(hn_dl):.4f}')

sbert_v5b.save(OUTPUT_V5B)
print(f'\nV5b salvat: {OUTPUT_V5B}')

GPU libera dupa curatare: 5.63 GB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fine-tuning V5b cu CE hard negatives:
  Start din: /kaggle/working/sbert_v5a
  batch=32 | triplete=39,606 | epoci=2


V5b Epoch 1/2:   0%|          | 0/1238 [00:00<?, ?it/s]

V5b Epoch 1 — Loss mediu: 1.4449


V5b Epoch 2/2:   0%|          | 0/1238 [00:00<?, ?it/s]

V5b Epoch 2 — Loss mediu: 0.7690


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


V5b salvat: /kaggle/working/sbert_v5b


In [18]:
sbert_v5b_eval = SentenceTransformer(OUTPUT_V5B, device=str(device))

print('Generare embeddings V5b...')
q_emb_v5b = sbert_v5b_eval.encode(queries_prefixed, batch_size=BATCH_EMB,
                                    show_progress_bar=True, convert_to_numpy=True,
                                    normalize_embeddings=True)
d_emb_v5b = sbert_v5b_eval.encode(docs, batch_size=BATCH_EMB,
                                    show_progress_bar=True, convert_to_numpy=True,
                                    normalize_embeddings=True)

np.save(OUTPUT_PATH + 'q_emb_v5b.npy', q_emb_v5b)
np.save(OUTPUT_PATH + 'd_emb_v5b.npy', d_emb_v5b)

print('\nEvaluare V5b full pool...')
results['v5b_full'] = evaluate_full_pool(q_emb_v5b, d_emb_v5b, label='v5b')
print(f'  {results["v5b_full"]}')

print('\nEvaluare V5b genre-filtered...')
results['v5b_genre'] = evaluate_genre_filtered(q_emb_v5b, d_emb_v5b)
print(f'  {results["v5b_genre"]}')

with open(OUTPUT_PATH + 'results_v5b.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nRezultate V5b salvate!')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generare embeddings V5b...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Evaluare V5b full pool...
  {1: 0.11281936462920118, 3: 0.18108316541035402, 5: 0.22158370027614002, 10: 0.2829315620568699, 20: 0.3491056546508446}

Evaluare V5b genre-filtered...
  Pool mediu per query: 15835 filme
  {1: 0.10033333333333333, 3: 0.14466666666666667, 5: 0.16833333333333333, 10: 0.20933333333333334, 20: 0.251}

Rezultate V5b salvate!


In [20]:
SAMPLE_BS  = 300
BATCH_BERT = 64

np.random.seed(42)
sample_idx = np.random.choice(len(df_valid), SAMPLE_BS, replace=False)

def get_top1_idx(q_emb, d_emb, idx):
    scores = cosine_similarity(q_emb[idx:idx+1], d_emb)[0]
    return int(np.argmax(scores))

def get_bert_text(idx):
    row = df_valid.iloc[idx]
    return clean_overview_summary(row['title'], row['overview_summary'])

refs      = [get_bert_text(i) for i in sample_idx]
hyps_base = [get_bert_text(get_top1_idx(q_emb_base, d_emb_base, i)) for i in sample_idx]
hyps_v5a  = [get_bert_text(get_top1_idx(q_emb_v5a,  d_emb_v5a,  i)) for i in sample_idx]
hyps_v5b  = [get_bert_text(get_top1_idx(q_emb_v5b,  d_emb_v5b,  i)) for i in sample_idx]

hit1_base = [1 if get_top1_idx(q_emb_base, d_emb_base, i) == i else 0 for i in sample_idx]
hit1_v5a  = [1 if get_top1_idx(q_emb_v5a,  d_emb_v5a,  i) == i else 0 for i in sample_idx]
hit1_v5b  = [1 if get_top1_idx(q_emb_v5b,  d_emb_v5b,  i) == i else 0 for i in sample_idx]

print(f'Sample BERTScore: {SAMPLE_BS} filme')
print(f'Hit@1 — base: {sum(hit1_base)/len(hit1_base):.3f} | v5a: {sum(hit1_v5a)/len(hit1_v5a):.3f} | v5b: {sum(hit1_v5b)/len(hit1_v5b):.3f}')

bert_results = {}
for name, hyps in [('base', hyps_base), ('v5a', hyps_v5a), ('v5b', hyps_v5b)]:
    print(f'\nBERTScore {name}...')
    _, _, F1 = bert_score_compute(
        hyps, refs, lang='en',
        model_type='bert-base-uncased',
        batch_size=BATCH_BERT, verbose=False
    )
    bert_results[name] = F1.numpy()
    hits_list = {'base': hit1_base, 'v5a': hit1_v5a, 'v5b': hit1_v5b}[name]
    f1_miss   = F1.numpy()[[h == 0 for h in hits_list]]
    print(f'  F1 global: {F1.mean():.4f} +- {F1.std():.4f}')
    if len(f1_miss) > 0:
        print(f'  F1 miss:   {f1_miss.mean():.4f}')

print('\nBERTScore calculat!')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sample BERTScore: 300 filme
Hit@1 — base: 0.097 | v5a: 0.130 | v5b: 0.067

BERTScore base...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1 global: 0.5161 +- 0.1936
  F1 miss:   0.4438

BERTScore v5b...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1 global: 0.4832 +- 0.1463
  F1 miss:   0.4463

BERTScore calculat!


In [21]:
cross_encoder = CrossEncoder(CE_MODEL, device=str(device))
print(f'Cross-encoder disponibil: {CE_MODEL}')


def evaluate_with_reranking(query_emb, doc_emb, queries_raw, docs_raw,
                             k_values=[1, 3, 5, 10], sample_n=500,
                             retrieval_k=50):
    np.random.seed(42)
    indices = np.random.choice(len(query_emb), min(sample_n, len(query_emb)), replace=False)
    hits    = {k: 0 for k in k_values}

    for idx in tqdm(indices, desc='Reranking eval'):
        scores    = cosine_similarity(query_emb[idx:idx+1], doc_emb)[0]
        top50     = np.argsort(scores)[::-1][:retrieval_k]
        query_raw = queries_raw[idx].replace(BGE_QUERY_PREFIX, '')
        pairs     = [(query_raw, docs_raw[j]) for j in top50]
        ce_scores = cross_encoder.predict(pairs, batch_size=32)
        reranked  = top50[np.argsort(ce_scores)[::-1]]

        for k in k_values:
            if idx in reranked[:k]:
                hits[k] += 1

    return {k: hits[k] / len(indices) for k in k_values}


print('\nEvaluare V5b + Cross-Encoder reranking (sample 500)...')
results['v5b_rerank'] = evaluate_with_reranking(
    q_emb_v5b, d_emb_v5b,
    queries_prefixed, docs,
    k_values=[1, 3, 5, 10]
)
print(f'  {results["v5b_rerank"]}')

with open(OUTPUT_PATH + 'results_final.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nRezultate finale salvate!')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-encoder disponibil: cross-encoder/ms-marco-MiniLM-L-12-v2

Evaluare V5b + Cross-Encoder reranking (sample 500)...


Reranking eval:   0%|          | 0/500 [00:00<?, ?it/s]

  {1: 0.126, 3: 0.176, 5: 0.192, 10: 0.23}

Rezultate finale salvate!
